In [ ]:
print('Running imports...')
import os
from captum.attr import IntegratedGradients

os.environ["MKL_THREADING_LAYER"] = "GNU"

import torch # long
from lightning.pytorch import seed_everything
import xarray as xr
from tqdm import tqdm
import pandas as pd
import numpy as np
import argparse
import sys 

sys.path.append("/home/rogui7909/code/predict_rain_norway/predict_rain")
from ml_module_rain.models.trained_models import load_trained_model

Running imports...


In [ ]:

def change_time_of_prediction_to_time_of_event(ds_in):
    steps = ds_in.timestep_future.size
    ds_out = xr.concat([ds_in.isel(timestep_future=k).shift(time_of_prediction=-(steps-k-1)) for k in range(ds_in.timestep_future.size)], dim='timestep_future')
    ds_out = ds_out.rename(time_of_prediction='time_of_event', timestep_future='timestep_past')
    ds_out = ds_out.assign_coords(timestep_past = np.arange(0,-ds_out.timestep_past.size,-1))
    ds_out = ds_out.sel(timestep_past=np.arange(-ds_out.timestep_past.size+1,1))
    return ds_out

def get_ds_with_predictions(dataloader, lightning_model):
    if torch.cuda.is_available():
        device="cuda"
    else:
        device='cpu'
    lightning_model.to(device)
    lightning_model.eval()

    all_targets = []
    all_preds = []

    with torch.no_grad():
        for x, y in tqdm(dataloader.val_loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            preds = torch.sigmoid(lightning_model(x))
            all_preds.append(preds)
            all_targets.append(y)

    # Concatenate on GPU, then move to CPU once
    preds = torch.cat(all_preds).cpu().numpy()
    targets = torch.cat(all_targets).cpu().numpy()

    ds_aligned = dataloader.ds_val.isel(time=range(targets.shape[0]))
    ds_aligned = ds_aligned.rename(time='time_of_prediction', timestep ='timestep_future')
    preds_da = xr.DataArray(preds, dims=["time_of_prediction", "timestep_future"], coords=ds_aligned.targets.coords)
    ds_aligned["predictions"] = preds_da
    
    # ds_out = change_time_of_prediction_to_time_of_event(ds_aligned[['predictions','targets','rain']])
    # ds_out_valid_times = ds_out.where(ds_out.targets.count('timestep_past') == ds_out.targets.count('timestep_past').max(), drop=True)
    # return ds_out_valid_times.transpose('time_of_event','timestep_past')
    
    ds_out = change_time_of_prediction_to_time_of_event(ds_aligned)
    ds_out_valid_times = ds_out.where(ds_out.targets.count('timestep_past') == ds_out.targets.count('timestep_past').max(), drop=True)
    return ds_out_valid_times.transpose('time_of_event','timestep_past','var_name','latitude','longitude')

In [ ]:
run_id = 'xppgsat1'
dataloader, lightning_model = load_trained_model(run_id)
# attrs =.xr.open_dataset('')

NameError: name 'load_trained_model' is not defined

In [ ]:
ds_val = get_ds_with_predictions(dataloader, lightning_model)


In [ ]:
from sklearn import sklearn.metrics.roc_auc_score